#### 1. Importing Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

sns.set_style('darkgrid')
plt.rcParams['font.size'] = 14
plt.rcParams['figure.figsize'] = (20, 6)
plt.rcParams['figure.facecolor'] = '#00000000'

#### 2. Loading the Dataset and Initial Exploration

In [ ]:
rain_data = pd.read_csv("../data/weatherAUS.csv")
rain_data

As you can see, the dataset contains 145,460 rows and 23 columns. Each row represents a daily weather observation for a specific location in Australia. The columns include various weather attributes such as temperature, humidity, wind speed, and whether it rained the next day (the target variable).

In [ ]:
rain_data.info()

Here is a brief overview of the dataset:
- 23 columns: 7 string columns, 16 numeric columns.
- The target variable is "RainTomorrow", which indicates whether it rained the next day (Yes/No).

In [ ]:
rain_data.isnull().sum()

This shows that there are missing values in several columns, with "Sunshine" having the highest number of missing values (69,835) and "MaxTemp" with lowest number of missing values (1,261). The presence of missing values will need to be addressed during the data cleaning and preprocessing stage.

#### 3. Data Cleaning and Preprocessing

In [ ]:
rain_data.dropna(subset=['RainTomorrow', 'RainToday'], inplace=True)

Here I have removed rows with missing values in the "RainToday" and "RainTomorrow" columns, as these are essential for our analysis and modeling. This will help ensure that we have a cleaner dataset to work with for our analysis and modeling. Now our column "RainTomorrow" and "RainToday" will not have any missing values, which is crucial for our target variable and one of the key features in our analysis.
Before: 145,460 rows
After: 140,787 rows
4673 rows with missing values in "RainTomorrow" or "RainToday" have been removed.

#### 4. Exploratory Data Analysis and Visualization

Now let me perform some EDA and visualization to understand the relationship between the features and the target variable "RainTomorrow". I will create some visualizations to explore the distribution of the target variable and how it relates to other features in the dataset. This will help us identify any patterns or trends that may be useful for our modeling.

Let me once see columns there for fast access and then I will start with the visualizations.

In [ ]:
rain_data.info()

### Location vs Rain days (RainToday)

In [ ]:
sns.histplot(
    data=rain_data,
    x='Location',
    hue='RainToday',
    multiple='stack',
    palette='Set2'
)
plt.xticks(rotation=90) # Rotate x-axis labels for better readability
plt.title('Distribution of RainToday by Location')
plt.xlabel('Location')
plt.ylabel('Count')
# plt.legend(title='RainToday', labels=['No', 'Yes'])
plt.show()

It is more interactive if I plot it with plotly

In [ ]:
px.histogram(
    data_frame=rain_data,
    x='Location',
    color='RainToday',
    title='Distribution of RainToday by Location',

    # labels={'Location': 'Location', 'count': 'Count'},
    # category_orders={'RainToday': ['No', 'Yes']}

)

From the histogram, we I can observe that certain locations have a higher frequency of rain days (RainToday = Yes) compared to others.

### Temp3pm vs RainTomorrow

In [ ]:
px.histogram(
    rain_data,
    x='Temp3pm',
    color='RainTomorrow',
    title='Distribution of Temperature at 3pm by RainTomorrow'
)

### RainTomorrow vs RainToday

In [ ]:
px.histogram(
    rain_data,
    x='RainTomorrow',
    color='RainToday',
    title='Distribution of RainTomorrow by RainToday'
)

### 

### Max_Temp vs Max_Temp

In [ ]:
px.scatter(
    rain_data,
    x = 'MinTemp',
    y = 'MaxTemp',
    color='RainToday',
    title = 'Min temp vs Max temp by RainToday'
)

In [ ]:
sns.histplot(
    rain_data,
    x = 'MinTemp',
    y = 'MaxTemp',
    hue = 'RainToday'
)
plt.title('Min temp vs Max temp by Rain Today')
plt.legend(title='RainToday', labels=['Yes', 'No'])
plt.show()

### Temperature at 3pm vs Humidity at 3pm 

In [ ]:
px.scatter(
    rain_data,
    x='Temp3pm',
    y='Humidity3pm',
    color='RainToday',
    title='Temperature at 3pm vs Humidity at 3pm by RainToday'
)

There so many other missing values in the dataset. So how can I train a model with missing values?
About 60% of the rows have missing values in at least one column. This is a significant portion of the dataset, and it can pose challenges for training a machine learning model.
To handle the missing values, we have a few options:
1. **Remove rows with missing values**: This is the simplest approach, but it can lead to a significant loss of data, especially if a large portion of the dataset has missing values.
2. **Impute missing values**: We can fill in the missing values using various imputation techniques, such as mean, median, mode, or more advanced methods like K-nearest neighbors (KNN) imputation or regression imputation.
3. **Use models that can handle missing values**: Some machine learning algorithms, such as decision trees and random forests, can handle missing values without the need for imputation.
Given the high percentage of missing values, it may be more appropriate to use imputation techniques or models that can handle missing values rather than removing rows, as this would allow us to retain more of the data for training our model.

### Train-Validate-Test

We have to split our dataset into train, validate and test data.

In [ ]:
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(rain_data, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42)

In [ ]:
print('train_df.shape: ', train_df.shape)
print('val_df.shape', val_df.shape)
print('test_df.shape', test_df.shape)

But as there is date in our above dataset, we have to split the data in a way that the training data comes before the validation and test data. This is important to prevent data leakage and ensure that our model is trained on past data and evaluated on future data.

In [ ]:
year = pd.to_datetime(rain_data.Date).dt.year
sns.countplot(
    x = year
)
plt.title('Number of Rows per Year')
plt.show()

In [ ]:
# count of rows per year
year.value_counts()

In [ ]:
train_df = rain_data[year < 2015]
validate_df = rain_data[year == 2015]
test_df = rain_data[year > 2015]


In [ ]:
print(train_df.shape)
print(validate_df.shape)
print(test_df.shape)

In [ ]:
train_df

In [ ]:
validate_df

In [ ]:
test_df

#### Input dataset and target variable

Not all columns are useful for training a model. In this case, the "Date" column may not be directly useful for training a model, as it is a temporal feature. However, we can extract useful features from the "Date" column, such as the year, month, day of the week, etc., which may help in improving the model's performance.

1. Extract input columns and target variable/column

In [ ]:
input_cols = list(rain_data.columns)[1:-1]
target_col = 'RainTomorrow'

In [ ]:
input_cols

In [ ]:
target_col

2. Extraction of input and output datasets

train data set

In [ ]:
train_input = train_df[input_cols].copy()
# train_target = (train_df[input_cols[-1]]).copy()
train_target = train_df[target_col].copy()

validation data set

In [ ]:
val_input = validate_df[input_cols].copy()
val_target = validate_df[target_col].copy()

test data set

In [ ]:
test_input = test_df[input_cols].copy()
test_target = test_df[target_col].copy()

**Now, let me put them one by one**

In [ ]:
train_input

In [ ]:
train_target

In [ ]:
val_input

In [ ]:
val_target

In [ ]:
test_input

In [ ]:
test_target

Let me check the rows whether there is any row where MinTem is greater than MaxTemp. If there is such row, then it is an error and we have to remove it from the dataset.

In [ ]:
check = train_input['MinTemp'] > train_input['MaxTemp']
train_input[check]

Here I found 6 rows where MinTemp is equal to MaxTemp.
Would this be an error? It is possible that there are days where the minimum and maximum temperatures are the same, especially if the temperature is constant throughout the day. However, it is also possible that these rows contain errors or outliers. To determine whether these rows should be removed, we can further investigate the data and check for any other anomalies or inconsistencies in those rows. If we find that these rows are indeed errors, we can consider removing them from the dataset to improve the quality of our data for modeling.

Let me focus on the training dataset for now. As it is used for training the model, it is important to know which columns are numeric and which are categorical, as this will affect how we preprocess the data and which algorithms we can use for modeling.

In [ ]:
numeric_cols = train_input.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_input.select_dtypes(exclude=np.number).columns.tolist()

Statistics of the numeric columns in the training dataset:

In [ ]:
train_input[numeric_cols].describe().transpose()
# train_input[numeric_cols].describe().T

Which categories are there in the categorical columns of the training dataset?

In [ ]:
train_input[categorical_cols].nunique()

#### Imputation of missing values

Before imputation, let me check how many numeric missing values there are.

In [ ]:
# in whole dataset
rain_data[numeric_cols].isna().sum()

In [ ]:
# in training dataset
train_input[numeric_cols].isna().sum()

Now let me impute the missing values using `SimpleImputer` with the strategy of mean imputation for numeric columns and mode imputation for categorical columns.

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
imputer.fit(rain_data[numeric_cols])

In [ ]:
print(imputer.statistics_)

Now I can impute the missing values in the numeric columns of train, validate and test datasets using the fitted imputer with transform method which is already fitted on the rain_df (whole dataset).
Why don't I fit the imputer again for each subset data?
The reason we fit the imputer on the whole dataset (rain_df) and then use the same fitted imputer to transform the train, validate, and test datasets is to ensure that the imputation is consistent across all subsets of the data. By fitting the imputer on the entire dataset, we are calculating the mean (or mode) values based on all available data, which provides a more accurate representation of the missing values.

In [ ]:
train_input[numeric_cols] = imputer.transform(train_input[numeric_cols])
val_input[numeric_cols] = imputer.transform(val_input[numeric_cols])
test_input[numeric_cols] = imputer.transform(test_input[numeric_cols])